# Les thèmes de la presse de 1914

De quoi parlait *L'Humanité* en 1914, et comment l'actualité a-t-elle basculé l'été venu ? Sans aucune étiquette, BERTopic regroupe les passages proches et décrit chaque groupe par ses mots les plus caractéristiques.

1. Trouver les thèmes
2. Les ajuster
3. Les suivre dans le temps
4. Les nommer avec un LLM

Les *embeddings* des passages sont déjà calculés (avec Qwen3-Embedding-0.6B, tronqués à 512 dimensions). Le GPU ne sert qu'au LLM de l'étape 4 : *Exécution > Modifier le type d'exécution > GPU T4*.

## Données

Ollama s'installe et télécharge le LLM de l'étape 4 en arrière-plan.

In [ ]:
!wget -q https://github.com/shuuchuu/datasets/raw/refs/heads/main/french-press-1914-passages/passages.parquet
!wget -q https://github.com/shuuchuu/datasets/raw/refs/heads/main/french-press-1914-passages/embeddings-512.npy
!apt-get -qq install -y zstd > /dev/null 2>&1
!curl -fsSL https://ollama.com/install.sh | sh > /dev/null 2>&1
%pip install -q bertopic

import os
import subprocess

MODEL = "qwen3:4b-instruct-2507-q4_K_M"
env = {**os.environ, "OLLAMA_NUM_PARALLEL": "8", "OLLAMA_KEEP_ALIVE": "-1"}
subprocess.Popen(["ollama", "serve"], env=env, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
pull = subprocess.Popen(f"sleep 5 && ollama pull {MODEL}", shell=True, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)

In [ ]:
import numpy as np
import pandas as pd

passages = pd.read_parquet("passages.parquet")
embeddings = np.load("embeddings-512.npy").astype(np.float32)

# L'Humanité seulement (Le Figaro n'a que 24 numéros), et un échantillon de 20 000
# passages pour que le calcul tienne en quelques minutes.
humanite = passages[passages.newspaper == "L'Humanité"].sample(20_000, random_state=0)
docs = humanite.text.tolist()
dates = humanite.date.tolist()
vectors = embeddings[humanite.index]
print(len(docs), vectors.shape)

## 1. Trouver les thèmes

Créez un modèle `BERTopic` avec :

- `umap_model=UMAP(n_neighbors=15, n_components=5, min_dist=0.0, metric="cosine", random_state=0)` (réduction à 5 dimensions, reproductible)
- `vectorizer_model=CountVectorizer(stop_words=STOP_WORDS, min_df=10)` : pour **décrire** les thèmes, les mots vides gênent (c'est le seul usage du cours où on les retire)
- `min_topic_size=50`
- `language="french"` : sinon, BERTopic retire des mots-clés tous les caractères accentués (« grève » devient « grve »)

Entraînez-le avec `fit_transform(docs, vectors)` (les vecteurs déjà calculés), puis affichez `get_topic_info()` : combien de thèmes ? Quelle part des passages est hors thème (thème $-1$) ? Lisez les mots-clés des 15 plus grands thèmes, et deux passages représentatifs (`get_representative_docs`) de trois d'entre eux.

In [ ]:
from bertopic import BERTopic
from sklearn.feature_extraction.text import CountVectorizer
from spacy.lang.fr.stop_words import STOP_WORDS
from umap import UMAP

STOP_WORDS = sorted(STOP_WORDS | {"qu", "mm", "mme"})

In [ ]:
# Votre code ici

### Solution

In [ ]:
topic_model = BERTopic(
    language="french",
    umap_model=UMAP(n_neighbors=15, n_components=5, min_dist=0.0, metric="cosine", random_state=0),
    vectorizer_model=CountVectorizer(stop_words=STOP_WORDS, min_df=10),
    min_topic_size=50,
)
topics, _ = topic_model.fit_transform(docs, vectors)
info = topic_model.get_topic_info()
print(len(info) - 1, "thèmes,", f"{np.mean(np.array(topics) == -1):.0%} hors thème")
info.head(16)

In [ ]:
for topic in info.Topic[1:4]:
    print(topic, topic_model.get_topic(topic)[:8])
    for doc in topic_model.get_representative_docs(topic)[:2]:
        print("   ", doc[:200])

## 2. Ajuster les thèmes

- Relancez avec `min_topic_size=20`, puis `200` : combien de thèmes, quelle part hors thème ?
- Avec le modèle de départ, réaffectez les passages hors thème à leur thème le plus proche (`topic_model.reduce_outliers(docs, topics, strategy="embeddings", embeddings=vectors)`), puis mettez à jour les descriptions (`topic_model.update_topics(docs, topics=new_topics, vectorizer_model=...)`).
- Repérez deux thèmes qui parlent de la même chose et fusionnez-les (`topic_model.merge_topics(docs, [a, b])`).

In [ ]:
# Votre code ici

### Solution

In [ ]:
for size in (20, 200):
    model = BERTopic(
        language="french",
        umap_model=UMAP(n_neighbors=15, n_components=5, min_dist=0.0, metric="cosine", random_state=0),
        vectorizer_model=CountVectorizer(stop_words=STOP_WORDS, min_df=10),
        min_topic_size=size,
    )
    found, _ = model.fit_transform(docs, vectors)
    print(f"min_topic_size={size} : {len(set(found)) - 1} thèmes, {np.mean(np.array(found) == -1):.0%} hors thème")

In [ ]:
new_topics = topic_model.reduce_outliers(docs, topics, strategy="embeddings", embeddings=vectors)
topic_model.update_topics(docs, topics=new_topics, vectorizer_model=CountVectorizer(stop_words=STOP_WORDS, min_df=10))
print(f"{np.mean(np.array(new_topics) == -1):.0%} hors thème après réaffectation")
topics = new_topics

# Par exemple, deux thèmes sur les grèves, ou sur le procès Caillaux : à choisir en lisant
# les mots-clés de get_topic_info().

## 3. Suivre les thèmes dans le temps

`topic_model.topics_over_time(docs, dates, nr_bins=52)` compte les passages de chaque thème par semaine ; `topic_model.visualize_topics_over_time(...)` les trace. Affichez les thèmes qui parlent de la guerre, de Jaurès et du procès Caillaux : écrivez une fonction `topic_with(word)` qui renvoie le thème dont la plus grande part des passages contient `word`. Que se passe-t-il entre juillet et août 1914 ?

In [ ]:
# Votre code ici

### Solution

In [ ]:
over_time = topic_model.topics_over_time(docs, dates, nr_bins=52)


def topic_with(word: str) -> int:
    mentions = pd.Series(topics)[[word in doc.lower() for doc in docs]].value_counts()
    share = (mentions / pd.Series(topics).value_counts()).drop(-1, errors="ignore")
    return int(share.idxmax())


chosen = [topic_with(word) for word in ("guerre", "jaurès", "caillaux")]
print(chosen)
topic_model.visualize_topics_over_time(over_time, topics=chosen)

# En mars, Mme Caillaux tue Gaston Calmette, le directeur du Figaro ; son procès occupe
# juillet. Jaurès est assassiné le 31 juillet. À partir d'août, la guerre et les récits du
# front dominent (regardez aussi le plus grand thème, celui des opérations militaires).

## 4. Nommer les thèmes avec un LLM

Des mots-clés décrivent un thème, mais ne le nomment pas. Pour chacun des 20 plus grands thèmes, donnez au LLM ses 10 mots-clés et ses 3 passages représentatifs (leurs 300 premiers caractères), et demandez-lui un nom court, de 3 à 6 mots, en français, au format `TopicName` (rappelez ce format dans le prompt : le modèle ne voit pas le schéma). Lancez les 20 demandes en parallèle (`ThreadPoolExecutor(8)`), installez les noms avec `topic_model.set_topic_labels(noms)` (un dictionnaire thème → nom), et affichez `get_topic_info()`. Les noms sont-ils justes ? Lequel corrigeriez-vous à la main ?

In [ ]:
pull.wait()  # le modèle est téléchargé

from concurrent.futures import ThreadPoolExecutor

from openai import OpenAI
from pydantic import BaseModel

client = OpenAI(base_url="http://localhost:11434/v1", api_key="ollama")
# Un premier appel charge le modèle sur le GPU, où il reste : avant les calculs en parallèle.
client.chat.completions.create(model=MODEL, messages=[{"role": "user", "content": "Bonjour"}], max_tokens=1)
print(MODEL, "prêt")


class TopicName(BaseModel):
    name: str

In [ ]:
# Votre code ici

### Solution

In [ ]:
def name_topic(topic: int) -> str:
    keywords = ", ".join(word for word, _ in topic_model.get_topic(topic)[:10])
    examples = "\n\n".join(doc[:300] for doc in topic_model.get_representative_docs(topic)[:3])
    response = client.chat.completions.parse(
        model=MODEL,
        messages=[{"role": "user", "content": (
            "Voici un thème de la presse française de 1914, trouvé automatiquement.\n"
            f"Ses mots-clés : {keywords}\n\nTrois passages représentatifs :\n{examples}\n\n"
            "Donne-lui un nom court et précis, de 3 à 6 mots, en français.\n"
            # Le schéma contraint la sortie, mais le modèle ne le voit pas : sans ce rappel,
            # il lui arrive de s'égarer en espaces avant le JSON.
            'Réponds par un objet JSON : {"name": "..."}'
        )}],
        response_format=TopicName,
        temperature=0,
        max_tokens=100,
    )
    return response.choices[0].message.parsed.name


largest = [topic for topic in topic_model.get_topic_info().Topic if topic != -1][:20]
with ThreadPoolExecutor(8) as pool:
    names = dict(zip(largest, pool.map(name_topic, largest)))
topic_model.set_topic_labels(names)
topic_model.get_topic_info()[["Topic", "Count", "CustomName", "Representation"]].head(21)

# BERTopic sait aussi le faire seul : representation_model=bertopic.representation.OpenAI(client,
# model=MODEL), avec n'importe quel client compatible OpenAI. Les noms se relisent toujours :
# un LLM nomme bien un thème net, mal un thème fourre-tout.